In [1]:
import pandas as pd
import numpy as np

In [2]:
df_clean = pd.read_csv(r"../data/processed/clean_sales.csv")
df_clean["Date"] = pd.to_datetime(df_clean["Date"])
df_clean["StateHoliday"] = df_clean["StateHoliday"].astype(str)
df_clean.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_8988\3502285958.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_clean = pd.read_csv(r"../data/processed/clean_sales.csv")


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,Id,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-17,4852,519,1,1,0,0,303445,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No Promo
1,2,5,2015-07-17,4518,495,1,1,0,1,959585,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-17,6679,673,1,1,0,1,739744,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-17,10514,1343,1,1,0,1,864001,c,c,620.0,9.0,2009.0,0,0.0,0.0,No Promo
4,5,5,2015-07-17,4355,513,1,1,0,1,981931,a,a,29910.0,4.0,2015.0,0,0.0,0.0,No Promo


In [3]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830918 entries, 0 to 830917
Data columns (total 19 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      830918 non-null  int64         
 1   DayOfWeek                  830918 non-null  int64         
 2   Date                       830918 non-null  datetime64[ns]
 3   Sales                      830918 non-null  int64         
 4   Customers                  830918 non-null  int64         
 5   Open                       830918 non-null  int64         
 6   Promo                      830918 non-null  int64         
 7   StateHoliday               830918 non-null  object        
 8   SchoolHoliday              830918 non-null  int64         
 9   Id                         830918 non-null  int64         
 10  StoreType                  830918 non-null  object        
 11  Assortment                 830918 non-null  object  

In [4]:
df_fe = df_clean.copy()

In [5]:
df_fe = df_fe.sort_values(["Store", "Date"])
df_fe.reset_index(drop=True, inplace=True)
df_fe.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,Id,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,3,2013-01-02,5530,668,1,0,0,1,988300,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No Promo
1,1,4,2013-01-03,4327,578,1,0,0,1,919910,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No Promo
2,1,5,2013-01-04,4486,619,1,0,0,1,662609,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No Promo
3,1,6,2013-01-05,4997,635,1,0,0,1,1008092,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No Promo
4,1,1,2013-01-07,7176,785,1,1,0,1,935813,c,a,1270.0,9.0,2008.0,0,0.0,0.0,No Promo


In [6]:
df_fe["Year"] = df_fe["Date"].dt.year
df_fe["Month"] = df_fe["Date"].dt.month
df_fe["Quarter"] = df_fe["Date"].dt.quarter
df_fe["Week"] = df_fe["Date"].dt.isocalendar().week.astype(int)
df_fe["Day"] = df_fe["Date"].dt.day
df_fe["DayOfYear"] = df_fe["Date"].dt.dayofyear
df_fe["WeekOfYear"] = df_fe["Date"].dt.isocalendar().week.astype(int)

In [7]:
df_fe["IsWeekend"] = (
    df_fe["DayOfWeek"].isin([6,7])
).astype(int)

In [8]:
df_fe["IsMonthStart"] = df_fe["Date"].dt.is_month_start.astype(int)

df_fe["IsMonthEnd"] = df_fe["Date"].dt.is_month_end.astype(int)

In [9]:
df_fe["Lag_1"] = (
    df_fe.groupby("Store")["Sales"].shift(1)
)

df_fe["Lag_7"] = (
    df_fe.groupby("Store")["Sales"].shift(7)
)

df_fe["Lag_14"] = (
    df_fe.groupby("Store")["Sales"].shift(14)
)

df_fe["Lag_30"] = (
    df_fe.groupby("Store")["Sales"].shift(30)
)

In [10]:
df_fe["RollingMean_7"] = (
    df_fe.groupby("Store")["Sales"].transform(lambda x: x.shift(1).rolling(7).mean())
)

df_fe["RollingMean_14"] = (
    df_fe.groupby("Store")["Sales"].transform(lambda x: x.shift(1).rolling(14).mean())
)

df_fe["RollingMean_30"] = (
    df_fe.groupby("Store")["Sales"].transform(lambda x: x.shift(1).rolling(30).mean())
)

In [11]:
df_fe["RollingStd_7"] = (
    df_fe.groupby("Store")["Sales"]
         .transform(lambda x: x.shift(1).rolling(7).std())
)

In [12]:
competition_start = pd.to_datetime(
    dict(
        year=df_fe["CompetitionOpenSinceYear"].replace(0,np.nan),
        month=df_fe["CompetitionOpenSinceMonth"].replace(0,np.nan),
        day=1
    ),
    errors="coerce"
)
competition_start

0        2008-09-01
1        2008-09-01
2        2008-09-01
3        2008-09-01
4        2008-09-01
            ...    
830913          NaT
830914          NaT
830915          NaT
830916          NaT
830917          NaT
Length: 830918, dtype: datetime64[ns]

In [13]:
df_fe["CompetitionAgeMonths"] = (
    (df_fe["Date"] - competition_start)
    .dt.days
    .div(30)
)

df_fe["CompetitionAgeMonths"] = (
    df_fe["CompetitionAgeMonths"]
    .clip(lower=0)
    .fillna(0)
)

In [14]:
promo_start = pd.to_datetime(
    df_fe["Promo2SinceYear"].replace(0,np.nan).astype("Int64").astype(str)
    + "-01-01",
    errors="coerce"
)

C:\Users\HP\AppData\Local\Temp\ipykernel_8988\1952205223.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  promo_start = pd.to_datetime(


In [15]:
promo_start += pd.to_timedelta(
    (df_fe["Promo2SinceWeek"]-1)*7,
    unit="D"
)

df_fe["PromoDurationMonths"] = (
    (df_fe["Date"]-promo_start)
    .dt.days
    .div(30)
)

df_fe["PromoDurationMonths"] = (
    df_fe["PromoDurationMonths"]
    .clip(lower=0)
    .fillna(0)
)

In [16]:
df_fe["Month_sin"] = np.sin(
    2*np.pi*df_fe["Month"]/12
)

df_fe["Month_cos"] = np.cos(
    2*np.pi*df_fe["Month"]/12
)

In [17]:
df_fe["DayOfWeek_sin"] = np.sin(
    2*np.pi*df_fe["DayOfWeek"]/7
)

df_fe["DayOfWeek_cos"] = np.cos(
    2*np.pi*df_fe["DayOfWeek"]/7
)

In [18]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "StoreType",
    "Assortment",
    "StateHoliday",
    "PromoInterval"
]

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df_fe[col] = le.fit_transform(df_fe[col].astype(str))
    encoders[col] = le

In [19]:
df_fe = df_fe.dropna().reset_index(drop=True)

In [20]:
df_fe.head(10)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,Id,...,RollingMean_7,RollingMean_14,RollingMean_30,RollingStd_7,CompetitionAgeMonths,PromoDurationMonths,Month_sin,Month_cos,DayOfWeek_sin,DayOfWeek_cos
0,1,3,2013-02-06,6140,693,1,1,0,0,226264,...,5388.428571,5346.071429,5103.833333,1109.382990,53.966667,0.0,0.866025,0.5,0.433884,-0.900969
1,1,4,2013-02-07,5499,675,1,1,0,0,318919,...,5733.428571,5399.357143,5124.166667,851.378658,54.000000,0.0,0.866025,0.5,-0.433884,-0.900969
2,1,5,2013-02-08,5681,630,1,1,0,0,440864,...,5861.714286,5383.571429,5163.233333,707.864797,54.033333,0.0,0.866025,0.5,-0.974928,-0.222521
3,1,6,2013-02-09,5370,656,1,0,0,0,779012,...,6000.571429,5390.928571,5203.066667,512.408319,54.066667,0.0,0.866025,0.5,-0.781831,0.623490
4,1,1,2013-02-11,4409,599,1,0,0,0,179504,...,5963.000000,5403.428571,5215.500000,551.966786,54.133333,0.0,0.866025,0.5,0.781831,0.623490
5,1,2,2013-02-12,4015,572,1,0,0,0,554905,...,5740.000000,5319.357143,5123.266667,805.684802,54.166667,0.0,0.866025,0.5,0.974928,-0.222521
6,1,3,2013-02-13,4252,604,1,0,0,0,274891,...,5309.000000,5206.285714,5071.100000,806.308667,54.200000,0.0,0.866025,0.5,0.433884,-0.900969
7,1,4,2013-02-14,4241,573,1,0,0,0,129150,...,5052.285714,5220.357143,5030.466667,817.428430,54.233333,0.0,0.866025,0.5,-0.433884,-0.900969
8,1,5,2013-02-15,4809,607,1,0,0,0,857234,...,4781.000000,5257.214286,5008.766667,703.446989,54.266667,0.0,0.866025,0.5,-0.974928,-0.222521
9,1,6,2013-02-16,6154,682,1,0,0,0,210939,...,4682.428571,5272.071429,5006.366667,630.644077,54.300000,0.0,0.866025,0.5,-0.781831,0.623490


In [21]:
df_fe.to_csv(
    "../data/processed/feature_engineered_data.csv",
    index=False
)

In [22]:
lag_df = df_fe[
    [
        "Store",
        "Date",
        "Sales"
    ]
]

lag_df.to_csv(
    "../data/processed/lag_history.csv",
    index=False
)